# Reproduce the Bastion Prompt Protection benchmark

Run the full **detection leaderboard** and **false-positive-rate** suite on a **free Colab T4** — roughly 15–30 minutes. Every number in the README and on the model card is produced by these two scripts, on public models and public benchmarks; this notebook lets you verify them yourself.

**Before you start:** set a GPU runtime — *Runtime → Change runtime type → T4 GPU*.

## 1. Get the code + install

In [ ]:
!git clone --quiet https://github.com/bastion-soft/bastion-prompt-protection.git
%cd bastion-prompt-protection
!pip install -q -e ".[eval]"

## 2. (Optional) Hugging Face login

A few baselines are **gated** and need a free HF token with access granted:
- `meta-llama/Prompt-Guard-86M` — accept on its model page
- `lmsys/lmsys-chat-1m` (an FPR dataset) — accept its license

Without a token they're **skipped cleanly** and the rest of the run continues. The commercial Bastion *multilingual* model also requires a license, so it is skipped here unless your token has been granted access.

In [ ]:
# Optional: add HF_TOKEN to Colab Secrets (key icon, left), or skip this cell.
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
    print("Logged in via Colab secret HF_TOKEN.")
except Exception:
    print("No HF_TOKEN found - gated models/datasets will be skipped (that's fine).")

## 3. Detection leaderboard

AUC + F1 across four held-out adversarial benchmarks (rogue, xTRam1, S-Labs, JailbreakBench). The script prints a markdown table and writes `eval/results/leaderboard.{json,md}`.

Tip: add `--limit 200` for a fast smoke run first.

In [ ]:
!python -m scripts.run_leaderboard

## 4. False-positive rate

The half most comparisons skip: the share of **benign** real-user messages each model wrongly flags, on WildChat + LMSYS openers. Lower is better. Writes `eval/results/false_positives.json`.

Tip: add `--n 500` for a fast smoke run first.

In [ ]:
!python -m scripts.measure_false_positives

## Results

- `eval/results/leaderboard.json` + `leaderboard.md` — detection (AUC / F1 / latency)
- `eval/results/false_positives.json` — false-positive rate

Full harness docs: [`eval/README.md`](https://github.com/bastion-soft/bastion-prompt-protection/blob/main/eval/README.md).